# 간단한 전이학습 시켜보기

1. 주제를 정한다.
2. 주제에 맞는 태스크를 판단한다.
3. 해당 태스크를 수행하는 모델을 huggingface에서 찾는다.
4. 적절한 데이터셋을 정제하여 준비한다.
5. 3에서 찾은 모델을 로드하여 '간단하게' 전이학습을 진행한다.
6. 추론 결과를 확인한다.

In [48]:
# from google.colab import drive
# drive.mount('/content/drive')

### 데이터 출처: https://www.kaggle.com/datasets/datasnaek/mbti-type
### 모델 출처: https://huggingface.co/facebook/nllb-200-distilled-600M

In [49]:
import numpy as np
import pandas as pd

mbti_df = pd.read_csv('/content/drive/MyDrive/전이학습/mbti_1.csv')
print(mbti_df.info())
print(mbti_df['posts'].value_counts())
mbti_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8675 entries, 0 to 8674
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   type    8675 non-null   object
 1   posts   8675 non-null   object
dtypes: object(2)
memory usage: 135.7+ KB
None
posts
'It has been too long since I have been on personalitycafe - although it doesn't seem to have changed one bit - but I must say it is good to be back somewhere like this. Usually I turn to Doctor Who...|||http://www.youtube.com/watch?v=6EEW-9NDM5k|||Overwhelmed by the world around me.|||In one dream I have had I was being chased by a large shadowy creature, with someone else who I felt I had to save above all else. The dream ended after she reached safety, but as for what happened...|||Well now My Avatar is a Doctor Who Clockwork Creature. I always liked this monster because It is just a worker trying to do it's job, kind of :3|||1st - Thanks for your reply, I appreciate all the help I can

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...
...,...,...
8670,ISFP,'https://www.youtube.com/watch?v=t8edHB_h908||...
8671,ENFP,'So...if this thread already exists someplace ...
8672,INTP,'So many questions when i do these things. I ...
8673,INFP,'I am very conflicted right now when it comes ...


In [50]:
import re, random

_URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.I)
_IMG_EXT_RE = re.compile(r"\b\w+\.(jpg|jpeg|png|gif|bmp|webp)\b", re.I)

def _clean_text(s: str) -> str:
    s = _URL_RE.sub("", str(s))
    s = _IMG_EXT_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace("’","'").replace("“","\"").replace("”","\"")
    return s

def _sentence_split(s: str):
    parts = re.split(r'(?<=[.!?])\s+', s)
    return [p.strip() for p in parts if len(p.split()) >= 3]

def _is_english_dominant(s: str, ratio: float = 0.6) -> bool:
    letters = [ch for ch in s if ch.isalpha()]
    if not letters:
        return False
    ascii_letters = [ch for ch in letters if ord(ch) < 128]
    return (len(ascii_letters) / len(letters)) >= ratio

def preprocess_posts(
    raw_text: str,
    sample_size: int = 3,
    eng_ratio: float = 0.6,
    seed: int = 0,
):
    random.seed(seed)

    parts = [p for p in str(raw_text).split("|||") if p and p.strip()]

    sentences = []
    for p in parts:
        cleaned = _clean_text(p)
        sentences.extend(_sentence_split(cleaned))

    sentences = [s for s in sentences if _is_english_dominant(s, eng_ratio)]

    if len(sentences) > sample_size:
        sentences = random.sample(sentences, sample_size)

    return sentences


sampled = preprocess_posts(mbti_df['posts'][0], sample_size=1)
print(len(sampled), "문장")
for s in sampled[:3]:
    print("-", s)

1 문장
- when your main social outlet is xbox live conversations and even then you verbally fatigue quickly.


In [51]:
from transformers import pipeline, AutoTokenizer

MODEL = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

translator = pipeline(
    "translation",
    model=MODEL,
    tokenizer=tokenizer,
    src_lang="eng_Latn",   # 영어 BCP-47
    tgt_lang="kor_Hang",   # 한국어 BCP-47
    max_length=1024
)

translator('hello, my name is choiwoojin. my hobby is play the FC-online.')

Device set to use cuda:0


[{'translation_text': '안녕하세요, 제 이름은 최우진입니다. 제 취미는 FC 온라인 게임을 하는 것입니다.'}]

In [52]:
ko_mbti_df = translator(preprocess_posts(mbti_df['posts'][0], sample_size=1))
ko_mbti_df

[{'translation_text': '여러분의 주요 소셜 네트워크는 Xbox 라이브 대화이고, 심지어는 말 그대로도 빨리 피곤해집니다.'}]

In [53]:
from tqdm import tqdm
import pandas as pd

translated_data = []

for i in tqdm(range(len(mbti_df['posts']))):
    preprocessed_sentences = preprocess_posts(mbti_df['posts'][i], sample_size=1)
    if not preprocessed_sentences:
        continue
    translated_sentences = translator(preprocessed_sentences)
    mbti_type = mbti_df['type'][i]

    for translated_sentence_dict in translated_sentences:
        translated_data.append({
            'post': translated_sentence_dict['translation_text'],
            'type': mbti_type
        })

translated_mbti = pd.DataFrame(translated_data)
translated_mbti.to_csv('translated_mbti_data.csv', index=False)

translated_mbti_df = pd.read_csv('translated_mbti_data.csv')
print(translated_mbti_df.head())

100%|██████████| 8675/8675 [45:56<00:00,  3.15it/s]


                                                post  type
0  여러분의 주요 소셜 네트워크는 Xbox 라이브 대화이고, 심지어는 말 그대로도 빨리...  INFJ
1  내 데스크톱에 있는 개인 주식은 무작위 주식 사이트와 주식 사진 부켓에서 다운로드 ...  ENTP
2                           나는 음식을 낭비하는 것을 좋아하지 않는다.  INTP
3                                          좋은 생각이었어!  INTJ
4                                              너무...  ENTJ
